# Generate QA Dataset for an Existing Corpus

Creates (or recreates) a `train_questions.parquet` for an **already-indexed**
corpus so it can be evaluated with `rag_evaluation.ipynb`.

**Workflow:**
1. Set `NAME` to the target collection (must already have `wiki_corpus.parquet`)
2. Choose QA sources & balancing options
3. Run all cells — loads, enriches, balances, saves
4. Open `rag_evaluation.ipynb` with the same `NAME` and evaluate

In [1]:
from pathlib import Path
import pandas as pd
from config import DATA_DIR, CACHE_DIR

# ── Target collection (must already have wiki_corpus.parquet) ────────────────
NAME = "wiki_500k"
COLLECTION_ROOT = Path(DATA_DIR) / NAME
WIKI_PARQUET   = COLLECTION_ROOT / "wiki_corpus.parquet"
QUESTIONS_PATH = COLLECTION_ROOT / "full_nq.parquet"

# ── QA sources (HuggingFace config names) ────────────────────────────────────
QA_DATASETS = ["natural_questions"]  # Datasets to pull questions from (must be in QUESTIONS_PATH)
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# ── Balancing ────────────────────────────────────────────────────────────────
BALANCE = True                               # Whether to balance questions across popularity deciles
TARGET_PER_DECILE = 8000              # None → downsample to smallest decile

# ── Synthetic generation (optional) ─────────────────────────────────────────
GENERATE_SYNTHETIC = False
QUESTIONS_PER_DECILE = 300
MODEL_NAME = "gpt-4.1-nano"

# ── Sanity check ─────────────────────────────────────────────────────────────
assert WIKI_PARQUET.exists(), f"Corpus not found: {WIKI_PARQUET}"
print(f"✓ Collection: {NAME}")
print(f"  Corpus:     {WIKI_PARQUET}  ({WIKI_PARQUET.stat().st_size / 1e9:.2f} GB)")
print(f"  QA sources: {QA_DATASETS}")
print(f"  Balance:    {BALANCE}  |  Synthetic: {GENERATE_SYNTHETIC}")

✓ Collection: wiki_500k
  Corpus:     /Users/cyro/Documents/VSC/PopularityBias/data/wiki_500k/wiki_corpus.parquet  (0.85 GB)
  QA sources: ['natural_questions']
  Balance:    True  |  Synthetic: False


In [2]:
from scripts.prepare_qa_dataset import prepare_qa_dataset

qa_df = prepare_qa_dataset(
    qa_datasets=QA_DATASETS,
    popularity_dataset=POPULARITY_DATASET,
    output_path=QUESTIONS_PATH,
    balance=BALANCE,
    target_per_decile=TARGET_PER_DECILE,
    generate_synthetic_flag=GENERATE_SYNTHETIC,
    corpus_path=WIKI_PARQUET,  # Always filter to corpus
    questions_per_decile=QUESTIONS_PER_DECILE,
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
)


📥 Loading QA datasets: ['natural_questions']


/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


  natural_questions: 81,533 questions
✓ Merged QA: 81,533 questions from 1 dataset(s)

📊 Combined QA: 81,533 questions

🔍 Filtering QA to match corpus...
  Removed 74,789 questions not in corpus (81,533 → 6,744)

📈 Enriching with popularity deciles...
Loading popularity data for decile calculation...
  Calculating global deciles...
✓ Enriched: 6,744 questions with deciles

⚖️ Balancing...
  Balancing to 8000 questions per decile...
✓ Balanced: 6,744 questions (8000 × 10 deciles)
  Distribution:
decile
2       1
3       3
4       7
5      12
6      26
7      81
8     296
9    6318
Name: count, dtype: int64

✅ Done: 6,744 questions saved to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_500k/full_nq.parquet
Distribution:
decile
2       1
3       3
4       7
5      12
6      26
7      81
8     296
9    6318
Name: count, dtype: int64
Sources:
dataset
natural_questions    6744
Name: count, dtype: int64


In [3]:
# ── Quick inspection ──────────────────────────────────────────────────────────
print(f"Saved: {QUESTIONS_PATH}")
print(f"Total: {len(qa_df):,} questions\n")

if "decile" in qa_df.columns:
    print("Per-decile distribution:")
    print(qa_df["decile"].value_counts().sort_index())

if "dataset" in qa_df.columns:
    print(f"\nSources:")
    print(qa_df["dataset"].value_counts())

display(qa_df.sample(5, random_state=42))

Saved: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_500k/full_nq.parquet
Total: 6,744 questions

Per-decile distribution:
decile
2       1
3       3
4       7
5      12
6      26
7      81
8     296
9    6318
Name: count, dtype: int64

Sources:
dataset
natural_questions    6744
Name: count, dtype: int64


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,is_synthetic,decile
3599,5326628078647952795,who has the primary responsibility for conduct...,"[individual states, ]",924170,Elections in the United States,28862.062500,68150.218750,natural_questions,False,9
1407,6478221480593769299,when did nelly hot in here come out,"[April 16 , 2002, ]",1439764,Hot in Herre,7290.708333,156667.364583,natural_questions,False,9
1550,-701468025081472517,when was the suez canal taken by egypt,"[18 July 1956, ]",29323,Suez Canal,166238.666667,6349.958333,natural_questions,False,9
3400,1706883873751813612,when did congress receive the resolution advoc...,"[June 28 , 1776, ]",31874,United States Declaration of Independence,134585.729167,4293.479167,natural_questions,False,9
2345,5406065403890306136,what is the opening part of the declaration of...,"[introduction, ]",31874,United States Declaration of Independence,134585.729167,4293.479167,natural_questions,False,9


In [4]:
# ── Verify overlap with corpus ────────────────────────────────────────────────
corpus_ids = set(pd.read_parquet(WIKI_PARQUET, columns=["wikipedia_id"])["wikipedia_id"].astype(int))
qa_ids     = set(qa_df["wikipedia_id"].astype(int))

in_corpus = qa_ids & corpus_ids
missing   = qa_ids - corpus_ids

print(f"QA doc IDs in corpus: {len(in_corpus):,} / {len(qa_ids):,}  ({100 * len(in_corpus) / len(qa_ids):.1f}%)")
if missing:
    print(f"⚠️  {len(missing):,} QA doc IDs NOT in corpus — these questions can never be answered correctly")
else:
    print("✓ All QA documents exist in the corpus")

QA doc IDs in corpus: 3,249 / 3,249  (100.0%)
✓ All QA documents exist in the corpus
